# Practical 5: Case Study — Word Embeddings using Word2Vec and GloVe

**Mapped Course Outcome:** CO2  
**BT Level Mapping:** BT3

## Aim

To train and use dense word embeddings and compare local Word2Vec embeddings with pretrained GloVe vectors for semantic-similarity analysis.

## Objectives

- Understand distributional word representation.
- Train a small Word2Vec model on a tokenized corpus.
- Query cosine similarity and nearest neighbours in embedding space.
- Load pretrained GloVe vectors and compare them with the local model.

## Tools / Requirements

- Python 3.10 or above
- Libraries: `gensim`, `numpy`
- Internet connection only for the first GloVe download
- Jupyter Notebook or Google Colab

If the imports fail, run the following in a notebook cell, then restart the kernel: `!pip install gensim numpy`

## Theory

A **word embedding** is a dense numeric vector representing a word. The distributional hypothesis states that words used in similar contexts tend to have similar meanings, so they should have nearby vectors. Unlike sparse one-hot vectors, embeddings can express graded semantic similarity.

**Word2Vec** learns vectors by predicting nearby words. Its CBOW architecture predicts a target word from its context, while Skip-gram predicts context words from a target word. **GloVe** learns vectors from global word co-occurrence statistics. A local Word2Vec model can capture the meaning used in a particular domain, but a small corpus gives unstable and limited results. Pretrained GloVe has broader general-language knowledge because it was trained on much more text.

Cosine similarity measures the angle between two vectors: `cos(a, b) = (a · b) / (||a|| ||b||)`. A value near 1 indicates similar vector directions; a value near 0 indicates little relation.

## Methodology

1. Build a small, already-tokenized corpus with food, transport, health, and technology contexts.
2. Train a reproducible Skip-gram Word2Vec model.
3. Inspect the vocabulary, cosine similarities, and nearest neighbours.
4. Load the pretrained `glove-wiki-gigaword-50` vectors through Gensim downloader.
5. Compare the local and pretrained results for the same words.

## Dataset

This in-code corpus is deliberately small and contains four topical groups. The word `apple` occurs only in food contexts, so the local model should learn its fruit sense. This makes a useful case study: local embeddings reflect the supplied context, whereas GloVe reflects general-language usage. For a real application, replace this corpus with a larger tokenized domain corpus such as product reviews, research abstracts, news, or clinical notes.

In [1]:
import numpy as np
from gensim.models import Word2Vec
import gensim.downloader as api

np.set_printoptions(precision=3, suppress=True)
print('Imports completed successfully.')

Imports completed successfully.


In [2]:
# Each inner list is one tokenized sentence.
corpus = [
    ['apple', 'banana', 'mango', 'orange', 'are', 'sweet', 'fruit'],
    ['fresh', 'apple', 'and', 'banana', 'make', 'a', 'healthy', 'fruit', 'salad'],
    ['mango', 'orange', 'and', 'banana', 'are', 'tropical', 'fruit'],
    ['fruit', 'market', 'sells', 'apple', 'banana', 'mango', 'and', 'orange'],
    ['car', 'bus', 'train', 'and', 'bicycle', 'are', 'transport', 'vehicles'],
    ['the', 'bus', 'and', 'train', 'carry', 'passengers', 'to', 'the', 'city'],
    ['a', 'car', 'uses', 'a', 'road', 'while', 'a', 'train', 'uses', 'a', 'rail'],
    ['doctor', 'nurse', 'and', 'patient', 'work', 'in', 'a', 'hospital'],
    ['the', 'doctor', 'treats', 'a', 'sick', 'patient', 'at', 'the', 'hospital'],
    ['a', 'nurse', 'cares', 'for', 'each', 'patient', 'in', 'the', 'hospital'],
    ['computer', 'software', 'code', 'and', 'data', 'support', 'technology'],
    ['a', 'programmer', 'writes', 'code', 'and', 'software', 'on', 'a', 'computer'],
    ['technology', 'uses', 'computer', 'data', 'and', 'software', 'systems'],
]

print(f'Number of sentences: {len(corpus)}')
print('Example sentence:', corpus[0])

Number of sentences: 13
Example sentence: ['apple', 'banana', 'mango', 'orange', 'are', 'sweet', 'fruit']


In [3]:
# sg=1 selects Skip-gram. workers=1 and seed make this small demonstration reproducible.
word2vec_model = Word2Vec(
    sentences=corpus,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    negative=5,
    epochs=500,
    seed=42,
    workers=1,
)

local_vectors = word2vec_model.wv
print(f'Vocabulary size: {len(local_vectors.key_to_index)}')
print('Vocabulary:', sorted(local_vectors.key_to_index))
print('Vector dimension:', local_vectors.vector_size)

Vocabulary size: 53
Vocabulary: ['a', 'and', 'apple', 'are', 'at', 'banana', 'bicycle', 'bus', 'car', 'cares', 'carry', 'city', 'code', 'computer', 'data', 'doctor', 'each', 'for', 'fresh', 'fruit', 'healthy', 'hospital', 'in', 'make', 'mango', 'market', 'nurse', 'on', 'orange', 'passengers', 'patient', 'programmer', 'rail', 'road', 'salad', 'sells', 'sick', 'software', 'support', 'sweet', 'systems', 'technology', 'the', 'to', 'train', 'transport', 'treats', 'tropical', 'uses', 'vehicles', 'while', 'work', 'writes']
Vector dimension: 50


In [4]:
def cosine_similarity(vector_a, vector_b):
    """Compute cosine similarity explicitly from two NumPy vectors."""
    denominator = np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    return float(np.dot(vector_a, vector_b) / denominator)

def similarity_or_na(embedding, word_a, word_b):
    """Return a similarity score, or None if either word is out of vocabulary."""
    if word_a not in embedding.key_to_index or word_b not in embedding.key_to_index:
        return None
    return float(embedding.similarity(word_a, word_b))

# Verify the library result using an explicit cosine-similarity calculation.
apple_vector = local_vectors['apple']
banana_vector = local_vectors['banana']
manual_score = cosine_similarity(apple_vector, banana_vector)
gensim_score = local_vectors.similarity('apple', 'banana')

print(f'Manual cosine similarity (apple, banana): {manual_score:.4f}')
print(f'Gensim similarity        (apple, banana): {gensim_score:.4f}')

Manual cosine similarity (apple, banana): 0.9939
Gensim similarity        (apple, banana): 0.9939


In [5]:
def print_similarity_table(embedding, pairs, heading):
    print(heading)
    print('-' * len(heading))
    for word_a, word_b in pairs:
        score = similarity_or_na(embedding, word_a, word_b)
        value = f'{score:.4f}' if score is not None else 'OOV'
        print(f'{word_a:>10} ↔ {word_b:<10}: {value}')

similarity_pairs = [
    ('apple', 'banana'),       # same local topic
    ('doctor', 'nurse'),       # same local topic
    ('car', 'train'),          # same local topic
    ('apple', 'computer'),     # different local topics
]

print_similarity_table(local_vectors, similarity_pairs, 'Local Word2Vec similarities')

Local Word2Vec similarities
---------------------------
     apple ↔ banana    : 0.9939
    doctor ↔ nurse     : 0.9895
       car ↔ train     : 0.9886
     apple ↔ computer  : 0.9081


In [6]:
def print_neighbours(embedding, word, topn=5):
    if word not in embedding.key_to_index:
        print(f'{word!r} is out of vocabulary.')
        return
    print(f'Nearest neighbours of {word!r}:')
    for neighbour, score in embedding.most_similar(word, topn=topn):
        print(f'  {neighbour:<12} {score:.4f}')

for query_word in ('apple', 'doctor', 'computer'):
    print_neighbours(local_vectors, query_word)
    print()

Nearest neighbours of 'apple':
  fruit        0.9958
  banana       0.9939
  orange       0.9919
  mango        0.9903
  tropical     0.9896

Nearest neighbours of 'doctor':
  at           0.9940
  treats       0.9910
  sick         0.9907
  hospital     0.9902
  work         0.9897

Nearest neighbours of 'computer':
  software     0.9951
  support      0.9929
  code         0.9917
  technology   0.9891
  data         0.9891



## Pretrained GloVe vectors

The next cell downloads `glove-wiki-gigaword-50` once and then reuses Gensim's local cache. It is intentionally protected with `try`/`except`: if the machine is offline, the Word2Vec case study still runs and the comparison cells explain that GloVe is unavailable.

In [7]:
GLOVE_MODEL_NAME = 'glove-wiki-gigaword-50'
glove_vectors = None

try:
    print(f'Loading {GLOVE_MODEL_NAME!r}. The first run may take a few minutes...')
    glove_vectors = api.load(GLOVE_MODEL_NAME)
    print(f'GloVe loaded: {len(glove_vectors.key_to_index):,} words, ' 
          f'{glove_vectors.vector_size} dimensions.')
except Exception as error:
    print('GloVe could not be loaded. Check the internet connection and rerun this cell.')
    print(f'Reason: {type(error).__name__}: {error}')

glove_available = glove_vectors is not None

Loading 'glove-wiki-gigaword-50'. The first run may take a few minutes...
[==================================================] 100.0% 66.0/66.0MB downloaded
GloVe loaded: 400,000 words, 50 dimensions.


In [8]:
comparison_pairs = [
    ('apple', 'banana'),
    ('doctor', 'nurse'),
    ('car', 'train'),
    ('apple', 'computer'),
]

print('Pair'.ljust(22), 'Local Word2Vec'.ljust(18), 'Pretrained GloVe')
print('-' * 62)
for word_a, word_b in comparison_pairs:
    local_score = similarity_or_na(local_vectors, word_a, word_b)
    glove_score = (similarity_or_na(glove_vectors, word_a, word_b)
                   if glove_available else None)
    local_text = f'{local_score:.4f}' if local_score is not None else 'OOV'
    glove_text = f'{glove_score:.4f}' if glove_score is not None else ('not loaded' if not glove_available else 'OOV')
    pair_label = word_a + ' + ' + word_b
    print(f'{pair_label:<22} {local_text:<18} {glove_text}')

Pair                   Local Word2Vec     Pretrained GloVe
--------------------------------------------------------------
apple + banana         0.9939             0.5608
doctor + nurse         0.9895             0.7977
car + train            0.9886             0.7658
apple + computer       0.9081             0.6403


In [9]:
if glove_available:
    print_neighbours(glove_vectors, 'apple')
    print()
    print_neighbours(glove_vectors, 'doctor')
else:
    print('Skip: GloVe neighbours require a successful GloVe download.')

Nearest neighbours of 'apple':
  blackberry   0.7543
  chips        0.7439
  iphone       0.7430
  microsoft    0.7334
  ipad         0.7331

Nearest neighbours of 'doctor':
  nurse        0.7977
  physician    0.7965
  patient      0.7612
  child        0.7559
  teacher      0.7538


## Results and Discussion

- The local Word2Vec model should place words from the same supplied group close together: for example, `apple` with fruit words and `doctor` with hospital words. Exact scores can vary slightly with library versions and training settings.
- Its small corpus means that the learned relationships are narrow and can be noisy. It has no knowledge of a word outside its vocabulary.
- GloVe normally gives broader neighbours and similarity relationships because it is pretrained on a large corpus. Its representation may differ from the local model: it captures common general-language senses, whereas local Word2Vec captures the meaning implied by the supplied corpus.
- Therefore, use local/domain-trained embeddings when specialised context matters and enough in-domain text is available; use pretrained embeddings when broad semantic coverage and a quick baseline are more important.

## Outcome

This practical demonstrates how to train a Word2Vec embedding model, calculate cosine similarity, retrieve nearest neighbours, and compare local embeddings with pretrained GloVe. It also shows why corpus size and domain strongly affect an embedding's usefulness.

## Viva Voce Questions and Answers

1. **What is a word embedding?** A dense real-valued vector that represents a word and can encode semantic or contextual relationships.
2. **Explain the distributional hypothesis.** Words occurring in similar contexts tend to have similar meanings.
3. **Differentiate one-hot vectors and dense embeddings.** One-hot vectors are sparse, high-dimensional, and do not express similarity; embeddings are compact, dense, and can express similarity through vector distance or angle.
4. **Differentiate CBOW and Skip-gram.** CBOW predicts a target word from surrounding words; Skip-gram predicts surrounding words from a target word.
5. **How does GloVe differ conceptually from Word2Vec?** GloVe learns from global co-occurrence statistics, while Word2Vec learns by local context-prediction objectives.
6. **What is cosine similarity?** A normalized dot product that measures the angle between vectors; higher values indicate more similar directions.
7. **Why is a large corpus important?** It provides more contexts and word occurrences, making vector estimates more reliable and covering more vocabulary.
8. **What is an out-of-vocabulary (OOV) word?** A word that does not exist in the embedding model's vocabulary, so the model cannot return its vector.
9. **When are domain-specific embeddings preferable?** When specialised senses or terminology matter, such as in medical, legal, financial, or product-specific text.
10. **Name two downstream NLP tasks that use embeddings.** Text classification, semantic search, named-entity recognition, sentiment analysis, and machine translation are examples.

## References

- [Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)
- [Stanford GloVe project](https://nlp.stanford.edu/projects/glove/)
- [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
- [Demonstrative video search: NPTEL Word2Vec GloVe word embeddings](https://www.youtube.com/results?search_query=NPTEL+Word2Vec+GloVe+word+embeddings)